# Cargador de Tablas — Microsoft Fabric Lakehouse

## Descripción

Lee los Parquets generados por `GeneradorDatosSinteticosPD.ipynb` y los registra
como Delta tables en el Lakehouse.

**Prerequisito**: ejecutar primero el generador para que los parquets existan.

**Plataforma**: ⚠️ Microsoft Fabric (Lakehouse) — requiere `mssparkutils`

---

## Parámetros Configurables (Celda 1)

| Parámetro | Tipo | Default | Propósito |
|-----------|------|---------|----------|
| `DESTINATION_SCHEMA` | str | `"dbo"` | Schema de destino en el Lakehouse. El campo `schema` del YAML lo sobreescribe por dataset. |
| `WRITE_MODE` | str | `"overwrite"` | `"overwrite"` reemplaza la tabla completa. `"append"` agrega filas. |
| `BASE_OUTPUT_PATH` | str | `"Files"` | Debe coincidir con el valor usado en el generador. |
| `YAML_FILES` | list | — | Misma lista que en el generador, en el mismo orden. |

---

## Schema por dataset

Cada YAML puede declarar su propio schema de destino con el campo `schema`:

```yaml
dataset:
  name: employees
  schema: HR          # ← este valor tiene prioridad sobre DESTINATION_SCHEMA
  output: "Files/Data/HR"
```

Si el YAML no tiene `schema`, se usa `DESTINATION_SCHEMA`.

---

**Versión**: 1.0
**Autor**: Javier Loria
**Licencia**: MIT — ver LICENSE

In [ ]:
# ============================================================================
# CONFIGURACIÓN (Personalizar aquí)
# ============================================================================

# Schema de destino en el Lakehouse.
# El campo 'schema' de cada YAML sobreescribe este valor para ese dataset.
DESTINATION_SCHEMA = "dbo"

# Modo de escritura:
#   "overwrite" → reemplaza la tabla completa (seguro para datos sintéticos)
#   "append"    → agrega filas sin borrar las existentes
WRITE_MODE = "overwrite"

# Ruta base del Lakehouse (debe coincidir con BASE_OUTPUT_PATH del generador)
BASE_OUTPUT_PATH = "Files"

# Lista de YAMLs a cargar — misma lista que en el generador
YAML_FILES = [
    "Files/config/employees.yml",
    "Files/config/products_catalog.yml",
    "Files/config/support_tickets.yml",
    "Files/config/iot_readings.yml",
]

print(f"🎯 Schema destino: {DESTINATION_SCHEMA}")
print(f"📝 Modo: {WRITE_MODE}")
print(f"📦 Datasets a cargar: {len(YAML_FILES)}")

In [ ]:
import yaml
import traceback
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CargadorTablas").getOrCreate()

def load_yaml_config(yaml_path):
    """Lee un YAML de configuración del Lakehouse."""
    if not yaml_path:
        raise ValueError("yaml_path no puede ser None o vacío")
    try:
        text = mssparkutils.fs.head(yaml_path, 10_000_000)
        config = yaml.safe_load(text)
    except Exception as e:
        raise FileNotFoundError(f"No se pudo leer {yaml_path}: {str(e)}")

    if config is None:
        raise ValueError(f"YAML vacío o inválido: {yaml_path}")
    if 'dataset' not in config:
        raise ValueError("YAML debe contener sección 'dataset'")

    return config['dataset']

print("✅ load_yaml_config definida")

In [ ]:
def load_parquet_to_table(config, destination_schema, write_mode):
    """
    Lee el parquet generado y lo registra como Delta table.

    Schema de destino: el campo 'schema' del YAML tiene prioridad sobre
    destination_schema. Si ninguno está definido, usa 'dbo'.
    """
    name = config['name']
    output_path = config.get('output', f"Files/Data/{name}")

    if not output_path.startswith('Files/'):
        output_path = f"Files/{output_path}"

    parquet_path = f"{output_path}/{name}.parquet"

    # Schema: el YAML sobreescribe el global
    schema = config.get('schema') or destination_schema or 'dbo'
    full_table_name = f"{schema}.{name}"

    print(f"\n📥 Leyendo: {parquet_path}")
    print(f"   → Tabla destino: {full_table_name}  (modo: {write_mode})")

    df = spark.read.parquet(parquet_path)
    row_count = df.count()

    df.write.format("delta").mode(write_mode).saveAsTable(full_table_name)

    print(f"   ✓ {row_count:,} filas → {full_table_name}")
    return full_table_name, row_count


def run_batch_load(yaml_files, destination_schema=None, write_mode=None):
    """Carga en batch todos los parquets de la lista de YAMLs."""
    # Toma los valores de configuración global si no se pasan explícitamente
    dest_schema = destination_schema or DESTINATION_SCHEMA
    mode = write_mode or WRITE_MODE

    print(f"\n{'='*80}")
    print(f"🚀 CARGA: {len(yaml_files)} datasets → schema '{dest_schema}' (modo: {mode})")
    print(f"{'='*80}\n")

    results = {}

    for idx, yaml_file in enumerate(yaml_files, 1):
        print(f"\n{'─'*80}")
        print(f"[{idx}/{len(yaml_files)}] {yaml_file}")
        print(f"{'─'*80}")

        dataset_name = yaml_file  # Fallback si load_yaml_config falla
        try:
            config = load_yaml_config(yaml_file)
            dataset_name = config['name']

            table_name, rows = load_parquet_to_table(config, dest_schema, mode)
            results[dataset_name] = {'status': 'OK', 'table': table_name, 'rows': rows}
            print(f"\n✅ Completado")

        except Exception as e:
            print(f"\n❌ ERROR: {str(e)}")
            traceback.print_exc()
            results[dataset_name] = {'status': 'ERROR', 'error': str(e)}

    # Resumen
    print(f"\n{'='*80}")
    print(f"📊 RESUMEN")
    print(f"{'='*80}")

    ok_count = error_count = 0
    for name, result in results.items():
        if result['status'] == 'OK':
            print(f"✅ {name}: {result['rows']:,} filas → {result['table']}")
            ok_count += 1
        else:
            print(f"❌ {name}: {result['error']}")
            error_count += 1

    print(f"\n📈 Total: {ok_count} ✅ | {error_count} ❌")
    return results

print("✅ Funciones load_parquet_to_table y run_batch_load definidas")

In [ ]:
# ============================================================================
# EJECUTAR
# ============================================================================

print("\n🎯 Iniciando carga...\n")
results = run_batch_load(YAML_FILES)
print(f"\n🎉 CARGA COMPLETADA")